# Análisis de Resultados — Segmentación Óptima de Contenido

Este notebook carga los resultados de los 45 experimentos y genera las tablas y gráficas para el informe técnico.

**Prerequisito:** Ejecutar primero el diseño experimental completo:
```bash
python src/main.py generate-instances
python src/main.py run-experiments
```

## 0. Importaciones y configuración

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

RESULTS_CSV = '../data/results/results_all.csv'
FIGURES_DIR = '../documentacion/figuras'
os.makedirs(FIGURES_DIR, exist_ok=True)

print('✓ Setup completo.')

## 1. Carga de resultados

In [ ]:
df = pd.read_csv(RESULTS_CSV)

# Añadir columna de tamaño de instancia
def get_size(name):
    if 'small' in name:  return 'small'
    if 'medium' in name: return 'medium'
    if 'large' in name:  return 'large'
    return 'unknown'

df['size'] = df['instance'].apply(get_size)

# Orden de categorías
size_order  = ['small', 'medium', 'large']
config_order = ['baseline_cosine', 'two_phase', 'dp_llm_full']
config_labels = {
    'baseline_cosine': 'A — Baseline (coseno)',
    'dp_llm_full':     'B — DP+LLM completo',
    'two_phase':       'C — Two-phase',
}

df['size'] = pd.Categorical(df['size'], categories=size_order, ordered=True)
df['config_label'] = df['config'].map(config_labels)

print(f'Total de filas: {len(df)}')
print(f'Configuraciones: {df["config"].unique().tolist()}')
print(f'Instancias únicas: {df["instance"].nunique()}')
df.head()

## 2. Tabla resumen por configuración y tamaño

In [ ]:
summary = df.groupby(['config', 'size'], observed=True).agg(
    n_corridas       = ('instance', 'count'),
    intra_coh_mean   = ('intra_coherence', 'mean'),
    intra_coh_std    = ('intra_coherence', 'std'),
    inter_sep_mean   = ('inter_separation', 'mean'),
    boundary_f1_mean = ('boundary_f1', 'mean'),
    time_mean_s      = ('time_total_s', 'mean'),
    llm_calls_total  = ('llm_calls', 'sum'),
).round(4)

print('=== TABLA RESUMEN ===')
print(summary.to_string())

## 3. Gráfica 1 — Coherencia interna por config y tamaño

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Usar solo configs disponibles
available_configs = [c for c in config_order if c in df['config'].values]
palette = sns.color_palette('muted', len(available_configs))

sns.barplot(
    data=df,
    x='size', y='intra_coherence',
    hue='config_label',
    order=size_order,
    capsize=0.1, errwidth=1.5,
    ax=ax, palette=palette
)

ax.set_title('Coherencia interna media por configuración y tamaño de instancia', fontweight='bold')
ax.set_xlabel('Tamaño de instancia')
ax.set_ylabel('Coherencia interna (↑ mejor)')
ax.legend(title='Configuración', loc='lower right')
ax.set_ylim(0, 1.05)

plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig1_intra_coherence.png', dpi=150)
plt.show()
print('✓ Guardado: fig1_intra_coherence.png')

## 4. Gráfica 2 — Boundary F1 por config y tamaño (instancias con ground truth)

In [ ]:
df_gt = df[df['has_ground_truth'] == True].copy()

if len(df_gt) > 0:
    fig, ax = plt.subplots(figsize=(11, 5))
    
    sns.barplot(
        data=df_gt,
        x='size', y='boundary_f1',
        hue='config_label',
        order=size_order,
        capsize=0.1, errwidth=1.5,
        ax=ax, palette=palette
    )
    
    ax.set_title('Boundary F1 (±1 tolerancia) por configuración y tamaño', fontweight='bold')
    ax.set_xlabel('Tamaño de instancia')
    ax.set_ylabel('Boundary F1 (↑ mejor)')
    ax.legend(title='Configuración')
    ax.set_ylim(0, 1.05)
    
    plt.tight_layout()
    fig.savefig(f'{FIGURES_DIR}/fig2_boundary_f1.png', dpi=150)
    plt.show()
    print('✓ Guardado: fig2_boundary_f1.png')
else:
    print('⚠️  No hay instancias con ground truth en los resultados.')

## 5. Gráfica 3 — Calidad vs. Tiempo (scatter)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

markers = {'small': 'o', 'medium': 's', 'large': '^'}
colors  = {c: p for c, p in zip(available_configs, palette)}

for config in available_configs:
    for size in size_order:
        subset = df[(df['config'] == config) & (df['size'] == size)]
        if len(subset) == 0: continue
        ax.scatter(
            subset['time_total_s'],
            subset['intra_coherence'],
            label=f'{config_labels[config]} ({size})',
            marker=markers[size],
            color=colors[config],
            s=80, alpha=0.8, edgecolors='white', linewidths=0.5
        )

ax.set_xscale('log')
ax.set_title('Calidad (coherencia interna) vs. Tiempo de ejecución', fontweight='bold')
ax.set_xlabel('Tiempo de ejecución (s) — escala logarítmica')
ax.set_ylabel('Coherencia interna (↑ mejor)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

plt.tight_layout()
fig.savefig(f'{FIGURES_DIR}/fig3_quality_vs_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Guardado: fig3_quality_vs_time.png')

## 6. Gráfica 4 — LLM calls vs. n por configuración

In [ ]:
df_llm = df[df['llm_calls'] > 0].copy()

if len(df_llm) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for config in [c for c in available_configs if c != 'baseline_cosine']:
        subset = df_llm[df_llm['config'] == config].sort_values('n')
        ax.scatter(subset['n'], subset['llm_calls'], 
                   label=config_labels[config], s=60, alpha=0.8)
        # Línea de tendencia
        if len(subset) > 2:
            z = np.polyfit(subset['n'], subset['llm_calls'], 2)
            p = np.poly1d(z)
            xs = np.linspace(subset['n'].min(), subset['n'].max(), 100)
            ax.plot(xs, p(xs), '--', alpha=0.5)
    
    ax.set_title('Número de llamadas al LLM vs. tamaño de instancia (n)', fontweight='bold')
    ax.set_xlabel('n (número de elementos)')
    ax.set_ylabel('Llamadas al LLM (↓ más eficiente)')
    ax.legend()
    
    plt.tight_layout()
    fig.savefig(f'{FIGURES_DIR}/fig4_llm_calls_vs_n.png', dpi=150)
    plt.show()
    print('✓ Guardado: fig4_llm_calls_vs_n.png')
else:
    print('⚠️  No hay corridas con llamadas LLM (solo baseline). Ejecuta dp_llm o two_phase.')

## 7. Tabla LaTeX para el informe

In [ ]:
# Genera una tabla resumen en formato Markdown para copiar al informe
table_md = summary.to_markdown()
print(table_md)

## 8. Análisis cualitativo: ejemplo de segmentación

In [ ]:
# Visualiza la segmentación de una instancia small específica
import sys; sys.path.insert(0, '..')
from src.instance import load_instance
from src.solver import baseline as solver_baseline
from src.solver.baseline import compute_embeddings, build_coherence_matrix_cosine

INSTANCE_PATH = '../data/instances/small_01.json'

try:
    problem = load_instance(INSTANCE_PATH)
    seg, stats = solver_baseline.solve(problem)
    
    print(f'Instancia: {problem.name} | n={problem.n}')
    print(f'Cortes predichos: {seg.cuts}')
    print(f'Cortes verdaderos: {problem.true_cuts}')
    print(f'Segmentos: {seg.num_segments()}')
    print()
    
    for i, (start, end) in enumerate(seg.segments()):
        texts = [problem.elements[j].text[:60] + '...' for j in range(start, end+1)]
        print(f'--- Segmento {i+1} (E[{start}..{end}]) ---')
        for t in texts:
            print(f'  • {t}')
        print()
except FileNotFoundError:
    print('⚠️  Genera las instancias primero: python src/main.py generate-instances')